# Solution F2 · GridSearchCV, RandomizedSearchCV and HalvingRandomSearchCV compared

**Type**: Compare

## Objective

Compare three hyperparameter search strategies for a gradient boosting by number of trainings, time, validation score and AUC on the
test set, and understand when random search and halving save effort without losing quality.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv              # noqa: F401
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, HalvingRandomSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(3, shuffle=True, random_state=0)
est = GradientBoostingClassifier(random_state=0)
grid = {"learning_rate": [0.03, 0.1, 0.3], "max_depth": [2, 3, 4], "n_estimators": [50, 100, 200], "subsample": [0.6, 1.0]}

## Commented solution

In [ ]:
searches = {
    "grid": GridSearchCV(est, grid, cv=cv, scoring="roc_auc"),
    "random": RandomizedSearchCV(est, grid, n_iter=12, cv=cv, scoring="roc_auc", random_state=0),
    "halving": HalvingRandomSearchCV(est, grid, n_candidates=27, factor=3, cv=cv, scoring="roc_auc", random_state=0),
}
table = pd.DataFrame(index=list(searches), columns=["n_trainings", "time_s", "best_score", "test_auc"], dtype=float)
for name, search in searches.items():
    t0 = time.time()
    search.fit(X_tr, y_tr)
    table.loc[name, "time_s"] = time.time() - t0
    table.loc[name, "n_trainings"] = len(search.cv_results_["params"]) * 3
    table.loc[name, "best_score"] = search.best_score_
    table.loc[name, "test_auc"] = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])
table.round(3)

In [ ]:
random_saving = 1 - table.loc["random", "n_trainings"] / table.loc["grid", "n_trainings"]
random_loss = table.loc["grid", "test_auc"] - table.loc["random", "test_auc"]
print(f"random search saves {random_saving:.0%} of the trainings and changes the test AUC by {-random_loss:+.4f}")

## Check

In [ ]:
with verify():
    check(set(table.index) == {"grid", "random", "halving"} and table.notna().all().all(), "complete table", "three rows and four columns with no NaN")
    check(table.loc["grid", "n_trainings"] == 54 * 3, "the grid does 54 x 3 = 162 trainings", "54 combinations x 3 folds")
    check(table.loc["random", "n_trainings"] == 12 * 3, "random search does 12 x 3 = 36", "12 combinations x 3 folds")
    check(table.loc["random", "time_s"] < table.loc["grid", "time_s"] / 2, "random search costs less than half of the grid in time", "36 trainings against 162")
    check(random_saving > 0.7 and abs(random_loss) < 0.03, "random search saves over 70% of the trainings without losing more than 0.03 of test AUC",
          "compare n_trainings and test_auc between grid and random")
    check(table.loc["halving", "n_trainings"] > table.loc["random", "n_trainings"] and table["test_auc"].min() > 0.72,
          "halving tries more combinations than random search (but on less data for the first ones), and all the strategies have a decent test AUC",
          "halving starts with 27 candidates and reduces them at every iteration")
done()

## What you learned

- The full grid does 162 trainings and random search 36 (−78%): yet the test AUC is practically the same. With few hyperparameters
  that really matter, a random sample of 12 combinations almost always finds a good region: it is the classic result of Bergstra and Bengio.
- **Halving** starts from many candidates (27) on little data and keeps only the best for successive iterations with more data: the count of
  trainings includes those done on subsamples, so it costs less than the apparent counts; it is particularly useful with slow models.
- The `best_score_` values are not comparable across strategies (halving evaluates the last candidates on more data): the right comparison is
  always the AUC on the test set, touched once.

## Common pitfall

Concluding that random search "is always better" from a single comparison. With many interacting hyperparameters, or a narrow and
badly defined search space, a fine grid can make the difference. The point is knowing how much budget you need, not believing a rule.

## Interview question

<details><summary>You have a budget of 100 trainings and 6 hyperparameters. Grid or random search?</summary>

Random search: with 6 hyperparameters a grid of 100 combinations would have about 2 values per parameter, that is almost no exploration, while 100
random samples test 100 different values of every parameter. If a few parameters matter much more than the others (as is almost always the case) random
search finds the good region with the same budget. Then I may refine with a narrow grid around the best, or use a
method with early discarding like halving.

</details>